# Disease Ortholog TF Expression Filter

Load disease orthologs, known TFs, and the expression summary; then build a final disease-TF gene list expressed above configurable cutoffs in at least `N` broad cell types.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR = Path("/Users/nick/Projects/data/morphseq/results/20260612")
OUT_DIR = Path("/Users/nick/Projects/repositories/morphseq/results/nlammers/20260612")

DISEASE_PATH = DATA_DIR / "gene2DiseaseViaOrthology_2026.05.21.csv"
EXPRESSION_PATH = DATA_DIR / "gene_expression_summary.csv"
TF_PATH = DATA_DIR / "zscapetools_tfs.csv"

# Common-sense defaults. Tune these and re-run from this cell downward.
MIN_CELL_TYPES = 3
MIN_AVG_EXPR = 1.0
MIN_FRAC_EXPR = 0.25
CELL_TYPE_COL = "cell_type_broad"

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 20)

In [ ]:
disease = pd.read_csv(DISEASE_PATH, skiprows=1, encoding="utf-8-sig")

# The current expression CSV has a truncated final line, so use the Python engine
# and skip malformed records. This retains all complete rows.
expression = pd.read_csv(EXPRESSION_PATH, engine="python", on_bad_lines="skip")

tfs = pd.read_csv(TF_PATH)

print(f"Disease ortholog table: {disease.shape[0]:,} rows x {disease.shape[1]:,} columns")
print(f"Expression summary: {expression.shape[0]:,} rows x {expression.shape[1]:,} columns")
print(f"Known TF table: {tfs.shape[0]:,} rows x {tfs.shape[1]:,} columns")

display(disease.head(3))
display(expression.head(3))
display(tfs.head())

In [ ]:
def clean_symbol(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.lower()


def add_ab_paralog_grouping(genes: pd.DataFrame) -> tuple[pd.DataFrame, set[str]]:
    genes = genes.copy()
    genes["paralog_suffix"] = genes["zf_symbol_key"].str.extract(r"([ab])$", expand=False)
    genes["paralog_stem"] = genes["zf_symbol_key"].str.replace(r"([ab])$", "", regex=True)

    suffix_counts = (
        genes.loc[genes["paralog_suffix"].notna()]
        .groupby("paralog_stem")["paralog_suffix"]
        .nunique()
    )
    ab_paralog_stems = set(suffix_counts[suffix_counts >= 2].index)

    genes["effective_gene_key"] = genes["zf_symbol_key"]
    genes["is_ab_paralog_group"] = genes["paralog_stem"].isin(ab_paralog_stems)
    genes.loc[genes["is_ab_paralog_group"], "effective_gene_key"] = genes.loc[
        genes["is_ab_paralog_group"], "paralog_stem"
    ]

    return genes, ab_paralog_stems


disease = disease.copy()
tfs = tfs.copy()

disease["zf_symbol_key"] = clean_symbol(disease["Zebrafish Gene Symbol"])
tfs["tf_symbol_key"] = clean_symbol(tfs["gene_short_name"])

disease_genes = (
    disease[["Zebrafish Gene ID", "Zebrafish Gene Symbol", "zf_symbol_key"]]
    .drop_duplicates()
    .sort_values(["Zebrafish Gene Symbol", "Zebrafish Gene ID"])
)
disease_genes, ab_paralog_stems = add_ab_paralog_grouping(disease_genes)

disease = disease.merge(
    disease_genes[
        [
            "Zebrafish Gene ID",
            "zf_symbol_key",
            "paralog_suffix",
            "paralog_stem",
            "effective_gene_key",
            "is_ab_paralog_group",
        ]
    ],
    on=["Zebrafish Gene ID", "zf_symbol_key"],
    how="left",
)

print(f"Total unique zebrafish disease ortholog genes: {disease_genes.shape[0]:,}")
print(f"Unique disease ortholog gene symbols: {disease_genes['zf_symbol_key'].nunique():,}")
print(
    "Effective disease genes after collapsing represented a/b paralog groups: "
    f"{disease_genes['effective_gene_key'].nunique():,}"
)
print(f"Represented a/b paralog groups collapsed: {len(ab_paralog_stems):,}")

In [ ]:
tf_lookup = tfs.rename(
    columns={
        "id": "tf_ensembl_id",
        "gene_short_name": "tf_gene_short_name",
        "family": "tf_family",
        "protein": "tf_protein",
        "entrez_id": "tf_entrez_id",
    }
)

disease_tf = disease.merge(
    tf_lookup,
    left_on="zf_symbol_key",
    right_on="tf_symbol_key",
    how="inner",
)

disease_tf_genes = (
    disease_tf[
        [
            "Zebrafish Gene ID",
            "Zebrafish Gene Symbol",
            "Human Ortholog Entrez Gene Id",
            "Human Ortholog Symbol",
            "tf_ensembl_id",
            "tf_gene_short_name",
            "tf_family",
            "tf_entrez_id",
            "zf_symbol_key",
            "paralog_suffix",
            "paralog_stem",
            "effective_gene_key",
            "is_ab_paralog_group",
        ]
    ]
    .drop_duplicates()
    .sort_values(["Zebrafish Gene Symbol", "tf_ensembl_id"])
)

print(f"Disease ortholog table rows after TF filter: {disease_tf.shape[0]:,}")
print(f"Unique disease TF genes by zebrafish symbol: {disease_tf['zf_symbol_key'].nunique():,}")
print(f"Unique disease TF Ensembl IDs: {disease_tf['tf_ensembl_id'].nunique():,}")
print(
    "Effective disease TF candidates after paralog collapse: "
    f"{disease_tf_genes['effective_gene_key'].nunique():,}"
)

display(disease_tf_genes.head(5))

In [ ]:
passing_expression = expression.loc[
    (expression["avg_expr"] >= MIN_AVG_EXPR)
    & (expression["frac_expr"] >= MIN_FRAC_EXPR),
    ["gene", CELL_TYPE_COL, "avg_expr", "frac_expr"],
]

passing_cell_type_counts = (
    passing_expression[["gene", CELL_TYPE_COL]]
    .drop_duplicates()
    .groupby("gene", as_index=False)
    .agg(n_passing_cell_types=(CELL_TYPE_COL, "nunique"))
)

expression_gene_stats = (
    expression.groupby("gene", as_index=False)
    .agg(
        max_avg_expr=("avg_expr", "max"),
        max_frac_expr=("frac_expr", "max"),
        n_observed_cell_types=(CELL_TYPE_COL, "nunique"),
    )
)

disease_tf_expression = (
    disease_tf_genes.merge(
        passing_cell_type_counts,
        left_on="tf_ensembl_id",
        right_on="gene",
        how="left",
    )
    .merge(
        expression_gene_stats,
        left_on="tf_ensembl_id",
        right_on="gene",
        how="left",
        suffixes=("", "_expr"),
    )
    .drop(columns=["gene", "gene_expr"])
)

disease_tf_expression["n_passing_cell_types"] = (
    disease_tf_expression["n_passing_cell_types"].fillna(0).astype(int)
)
MIN_CELL_TYPES=3
final_genes = (
    disease_tf_expression.query("n_passing_cell_types >= @MIN_CELL_TYPES")
    .sort_values(
        ["n_passing_cell_types", "max_avg_expr", "max_frac_expr", "Zebrafish Gene Symbol"],
        ascending=[False, False, False, True],
    )
    .reset_index(drop=True)
)
final_effective_candidates = (
    final_genes.drop_duplicates("effective_gene_key", keep="first")
    .reset_index(drop=True)
)

print(
    "Final disease TF genes passing expression cutoffs "
    f"(N >= {MIN_CELL_TYPES} cell types, avg_expr >= {MIN_AVG_EXPR}, "
    f"frac_expr >= {MIN_FRAC_EXPR}): {final_genes.shape[0]:,}"
)
print(f"Unique final zebrafish symbols: {final_genes['zf_symbol_key'].nunique():,}")
print(f"Unique final Ensembl IDs: {final_genes['tf_ensembl_id'].nunique():,}")
print(
    "Effective final disease TF candidates after paralog collapse: "
    f"{final_effective_candidates.shape[0]:,}"
)

display(final_effective_candidates.head(5))

In [ ]:
cutoff_tag = (
    f"N{MIN_CELL_TYPES}_avg{MIN_AVG_EXPR:g}_frac{MIN_FRAC_EXPR:g}"
    .replace(".", "p")
)
output_path = OUT_DIR / f"disease_tf_expression_filtered_{cutoff_tag}.csv"
effective_output_path = OUT_DIR / f"disease_tf_expression_filtered_effective_{cutoff_tag}.csv"

final_genes.to_csv(output_path, index=False)
final_effective_candidates.to_csv(effective_output_path, index=False)
print(f"Saved final gene list to: {output_path}")
print(f"Saved paralog-collapsed final candidate list to: {effective_output_path}")